<a href="https://colab.research.google.com/github/Veniveni123/CSA47-DEEP-LEARNING/blob/main/LAB%2011-20/Experiment%2018%3A%20RNN%2C%20LSTM%20and%20GRU%20Hyperparameter%20Experimentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q keras-tuner
import time
import numpy as np
import pandas as pd
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

vocab_size, max_len = 5000, 80
(X_tr, y_tr), (X_te, y_te) = keras.datasets.imdb.load_data(num_words=vocab_size)
X_tr = keras.preprocessing.sequence.pad_sequences(X_tr[:2000], maxlen=max_len)
y_tr = y_tr[:2000]
X_te = keras.preprocessing.sequence.pad_sequences(X_te[:800], maxlen=max_len)
y_te = y_te[:800]

rows = []
for cell in ['RNN', 'LSTM', 'GRU']:
    def build_model(hp):
        emb = hp.Choice('emb_dim', [32, 64])
        units = hp.Choice('units', [32, 64])
        drop = hp.Choice('dropout', [0.2, 0.4])
        lr = hp.Choice('lr', [1e-3, 3e-3])

        m = keras.Sequential([keras.layers.Embedding(vocab_size, emb, input_length=max_len)])
        if cell == 'RNN':
            m.add(keras.layers.SimpleRNN(units, dropout=drop))
        elif cell == 'LSTM':
            m.add(keras.layers.LSTM(units, dropout=drop))
        else:
            m.add(keras.layers.GRU(units, dropout=drop))
        m.add(keras.layers.Dense(1, activation='sigmoid'))
        m.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
        return m

    tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=2, overwrite=True, directory='kt_tmp', project_name=cell)
    t0 = time.time()
    tuner.search(X_tr, y_tr, validation_split=0.2, epochs=2, batch_size=64, verbose=0)
    elapsed = round(time.time() - t0, 2)

    hp = tuner.get_best_hyperparameters(1)[0]
    best = tuner.get_best_models(1)[0]
    acc = round(best.evaluate(X_te, y_te, verbose=0)[1], 4)

    rows.append({
        'Cell': cell,
        'Units': hp.get('units'),
        'Emb': hp.get('emb_dim'),
        'Dropout': hp.get('dropout'),
        'LR': hp.get('lr'),
        'Test Acc': acc,
        'Params': best.count_params(),
        'Time (s)': elapsed
    })

print(pd.DataFrame(rows).to_string(index=False))